# M9 — Final Kaggle submission pipeline

**Purpose.** Create a competition-valid `answer.csv` on Kaggle from the
retained M1 + M2 + multilingual-E5 policy. The final score is supplied
only by the competition evaluator; this notebook focuses on correct,
reproducible candidate generation and submission validation.

The fixed policy from M6 is: nearest-history quota 20 → E5 quota 10 →
M1 RRF candidates until 50.


## Plan

1. **Kaggle setup** — install missing packages without replacing the
   compiled Kaggle stack, read live ClearML Secrets, and locate Input.
2. **Candidate sources** — recreate M1 lexical, M2 history and E5
   dense retrieval inside category partitions.
3. **Fusion and validation** — apply frozen quotas and write the exact
   two-column CSV required by the evaluator.
4. **Output** — persist source candidates, manifest, timings and a ZIP
   under `/kaggle/working` for download after Save Version.


## 1. Kaggle environment, Input and live ClearML

Attach one Kaggle Dataset that contains exactly `train.parquet`,
`benchmark_queries.parquet` and `benchmark_items.parquet`. Enable a GPU
accelerator and Internet if the E5 weights are not already cached.

In **Add-ons → Secrets**, add `CLEARML_API_ACCESS_KEY` and
`CLEARML_API_SECRET_KEY`; add host secrets too only for a non-default
ClearML server. The values are read but never printed.


In [ ]:
# Do not use --upgrade or --no-deps. Kaggle already provides compiled
# NumPy/PyTorch/scikit-learn packages; normal pip resolution adds only missing
# packages such as pathlib2 required by ClearML.
!pip install -q "clearml>=1.16,<2.0" "sentence-transformers>=3.4,<4.0" "transformers>=4.51,<5.0" "snowballstemmer>=2.2"


In [ ]:
from __future__ import annotations

import gc
import json
import os
import re
import shutil
import time
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import snowballstemmer
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer

SEED = 42
TOP_K = 200
FINAL_K = 50
HISTORY_QUOTA = 20
DENSE_QUOTA = 10
RRF_K = 60
BM25_K1, BM25_B = 1.5, 0.75
CHAR_MAX_FEATURES = 200_000
CHAR_BATCH_SIZE = 32
DESCRIPTION_CHAR_LIMIT = 1_500
DENSE_BATCH_SIZE = 48
DENSE_SCORE_BATCH_SIZE = 128
ZERO_SHOT_E5_MODEL_ID = "intfloat/multilingual-e5-large-instruct"
E5_QUERY_INSTRUCTION = "Given a Russian service-search request, retrieve relevant service listings."

np.random.seed(SEED)
NOTEBOOK_STARTED = time.perf_counter()
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.pop("CLEARML_OFFLINE_MODE", None)


def load_dotenv(path: Path) -> None:
    """Load a repository-local .env without printing or overwriting secrets."""
    for raw_line in path.read_text(encoding="utf-8").splitlines():
        line = raw_line.strip()
        if line and not line.startswith("#") and "=" in line:
            key, value = line.split("=", 1)
            os.environ.setdefault(key.strip(), value.strip())


def safe_top_k(scores: np.ndarray, k: int) -> np.ndarray:
    """Return descending top-k positions; also supports a small category."""
    if len(scores) == 0:
        return np.empty(0, dtype=np.int64)
    k = min(k, len(scores))
    positions = np.argpartition(scores, len(scores) - k)[len(scores) - k :]
    return positions[np.argsort(scores[positions])[::-1]]


def canonical_query_frame(frame: pd.DataFrame) -> pd.DataFrame:
    """Canonical form used only to deduplicate historical query contexts."""
    result = frame[SEARCH_COLUMNS].copy()
    for column in ("search_query", "search_infm_params_text"):
        result[column] = (
            result[column]
            .astype("string")
            .fillna("<NA>")
            .str.lower()
            .str.strip()
            .str.replace(r"\s+", " ", regex=True)
        )
    for column in ("search_location_id", "search_is_delivery_search", "search_category"):
        result[column] = result[column].astype("string").fillna("<NA>")
    return result


SEARCH_COLUMNS = [
    "search_query",
    "search_location_id",
    "search_is_delivery_search",
    "search_infm_params_text",
    "search_category",
]
TRAIN_COLUMNS = [*SEARCH_COLUMNS, "item_id"]
ITEM_COLUMNS = [
    "item_id",
    "item_title_raw",
    "item_description_raw",
    "item_infm_params_text",
    "item_category_id",
]


from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()


def get_required_secret(name: str) -> str:
    try:
        value = secrets.get_secret(name)
    except Exception as exc:
        raise RuntimeError(f"Add Kaggle Secret {name!r} and grant this notebook access to it.") from exc
    if not value:
        raise RuntimeError(f"Kaggle Secret {name!r} is empty.")
    return value


def get_optional_secret(name: str) -> None:
    try:
        value = secrets.get_secret(name)
    except Exception:
        return
    if value:
        os.environ[name] = value


os.environ["CLEARML_API_ACCESS_KEY"] = get_required_secret("CLEARML_API_ACCESS_KEY")
os.environ["CLEARML_API_SECRET_KEY"] = get_required_secret("CLEARML_API_SECRET_KEY")
for optional_name in ("CLEARML_API_HOST", "CLEARML_WEB_HOST", "CLEARML_FILES_HOST"):
    get_optional_secret(optional_name)

INPUT_ROOT = Path("/kaggle/input")


def find_input_file(filename: str) -> Path:
    matches = sorted(INPUT_ROOT.glob(f"**/{filename}"))
    if len(matches) != 1:
        raise RuntimeError(
            f"Attach exactly one Kaggle Input that contains {filename!r}; found: {[str(path) for path in matches]}"
        )
    return matches[0]


TRAIN_PATH = find_input_file("train.parquet")
BENCHMARK_QUERIES_PATH = find_input_file("benchmark_queries.parquet")
BENCHMARK_ITEMS_PATH = find_input_file("benchmark_items.parquet")

# To switch after E08: create a Kaggle Dataset from the checkpoint directory,
# attach it as an Input, set this flag to True and point this path at the
# SentenceTransformer directory (the one that contains modules.json).
USE_FINETUNED_E5 = False
FINETUNED_E5_PATH = Path("/kaggle/input/avito-e5-finetuned/checkpoint-best")
if USE_FINETUNED_E5 and not FINETUNED_E5_PATH.is_dir():
    raise FileNotFoundError(f"Fine-tuned checkpoint Input was not found: {FINETUNED_E5_PATH}")
MODEL_SOURCE = str(FINETUNED_E5_PATH) if USE_FINETUNED_E5 else ZERO_SHOT_E5_MODEL_ID
RUN_TAG = "finetuned_e5" if USE_FINETUNED_E5 else "zero_shot_e5"
OUTPUT_DIR = Path("/kaggle/working") / f"m9_final_submission__{RUN_TAG}"
HF_CACHE_DIR = Path("/kaggle/temp/hf-cache")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
HF_CACHE_DIR.mkdir(parents=True, exist_ok=True)
os.environ.setdefault("HF_HOME", str(HF_CACHE_DIR))

print({
    "train": str(TRAIN_PATH),
    "benchmark_queries": str(BENCHMARK_QUERIES_PATH),
    "benchmark_items": str(BENCHMARK_ITEMS_PATH),
    "output_dir": str(OUTPUT_DIR),
    "model_source": MODEL_SOURCE,
    "use_finetuned_e5": USE_FINETUNED_E5,
    "clearml_credentials_present": True,
})

REQUIRE_GPU = True
USE_ALL_VISIBLE_GPUS = True


The next cell starts a live ClearML task with manual logging. Automatic
notebook hooks remain disabled because they previously caused Kaggle
runs to stall; this does not change model or metric logging.


In [ ]:
from clearml import Task

clearml_task = Task.init(
    project_name="avito-retrieval",
    task_name=f"M9__final_m1_m2_e5__{RUN_TAG}__s{SEED}",
    reuse_last_task_id=False,
    # Manual logging avoids the prior Kaggle Jupyter-hook stall.
    auto_connect_arg_parser=False,
    auto_connect_frameworks={"detect_repository": False},
    auto_resource_monitoring=False,
    auto_connect_streams=False,
)
clearml_task.connect(
    {
        "stage": "M9_final_submission",
        "seed": SEED,
        "candidate_budget": FINAL_K,
        "retrieval_top_k": TOP_K,
        "category_rule": "item_category_id == search_category; full-corpus fallback when absent",
        "m1_source": "stemmed title+params+description BM25 + control title char-TFIDF; RRF k=60",
        "m2_source": "nearest historical query char-TFIDF",
        "history_quota": HISTORY_QUOTA,
        "dense_source": "multilingual-e5-large-instruct",
        "dense_quota": DENSE_QUOTA,
        "model_source": MODEL_SOURCE,
        "use_finetuned_e5": USE_FINETUNED_E5,
    },
    name="config",
)
clearml_logger = clearml_task.get_logger()
print({"clearml_task_id": clearml_task.id, "offline_mode": False})


## 2. Load data and category-safe corpus partitions

We use the whole train set as historical evidence and the whole
benchmark corpus as the candidate universe. An absent query category
is the only case that falls back to global search.


In [ ]:
load_started = time.perf_counter()
benchmark_queries = pd.read_parquet(BENCHMARK_QUERIES_PATH, columns=["query_id", *SEARCH_COLUMNS])
candidate_items = pd.read_parquet(BENCHMARK_ITEMS_PATH, columns=ITEM_COLUMNS).reset_index(drop=True)
train_pairs = pd.read_parquet(TRAIN_PATH, columns=TRAIN_COLUMNS)

assert benchmark_queries["query_id"].astype(str).is_unique
assert candidate_items["item_id"].astype(str).is_unique
assert len(benchmark_queries) > 0 and len(candidate_items) >= FINAL_K and len(train_pairs) > 0

item_ids = candidate_items["item_id"].astype(str).to_numpy()
item_id_set = set(item_ids)
all_item_indices = np.arange(len(candidate_items), dtype=np.int64)
category_to_indices = {
    str(category): group.index.to_numpy(dtype=np.int64)
    for category, group in candidate_items.groupby("item_category_id", sort=False)
}
category_to_item_ids = {
    str(category): set(group["item_id"].astype(str))
    for category, group in candidate_items.groupby("item_category_id", sort=False)
}
allowed_indices_by_query = [
    category_to_indices.get(str(category), all_item_indices)
    for category in benchmark_queries["search_category"]
]
category_fallback_queries = int(
    sum(str(category) not in category_to_indices for category in benchmark_queries["search_category"])
)

# The group ids only make each distinct full search context contribute once to
# historical support. They are not labels for benchmark queries.
train_canonical = canonical_query_frame(train_pairs)
benchmark_canonical = canonical_query_frame(benchmark_queries)
all_contexts = pd.concat([train_canonical, benchmark_canonical], ignore_index=True)
group_ids, _ = pd.factorize(pd.MultiIndex.from_frame(all_contexts), sort=False)
train_pairs["query_group"] = group_ids[: len(train_pairs)]
train_pairs["query_text_norm"] = train_canonical["search_query"].astype(str).to_numpy()
benchmark_queries["query_text_norm"] = benchmark_canonical["search_query"].astype(str).to_numpy()
benchmark_queries["category_key"] = benchmark_canonical["search_category"].astype(str).to_numpy()

history_pairs = train_pairs.loc[train_pairs["item_id"].astype(str).isin(item_id_set)].copy()
assert len(history_pairs) > 0
load_seconds = time.perf_counter() - load_started

print({
    "benchmark_queries": len(benchmark_queries),
    "candidate_items": len(candidate_items),
    "train_positive_pairs": len(train_pairs),
    "history_pairs_in_candidate_corpus": len(history_pairs),
    "category_fallback_queries": category_fallback_queries,
    "load_seconds": round(load_seconds, 2),
})


## 3. M1 lexical candidates

Rebuild stemmed full-text BM25 and control title char-TFIDF, then fuse
their top-200 category-restricted lists with reciprocal-rank fusion.


In [ ]:
TOKEN_PATTERN = r"(?u)\b[0-9a-zа-я]{2,}\b"
NON_WORD_RE = re.compile(r"[^0-9a-zа-я]+")
STEMMER = snowballstemmer.stemmer("russian")


def normalize_russian_text(value: object) -> str:
    text = "" if pd.isna(value) else str(value)
    return " ".join(NON_WORD_RE.sub(" ", text.lower().replace("ё", "е")).split())


def stem_russian_text(value: object) -> str:
    return " ".join(STEMMER.stemWords(normalize_russian_text(value).split()))


def compose_document_text(frame: pd.DataFrame, fields: tuple[str, ...]) -> pd.Series:
    text = frame[fields[0]].fillna("").astype(str)
    for field in fields[1:]:
        text = text.str.cat(frame[field].fillna("").astype(str), sep=" ")
    return text


class SparseBM25:
    """Exact Okapi BM25 over a sparse CSC matrix, as in the retained M1 source."""

    def __init__(self, k1: float = BM25_K1, b: float = BM25_B, epsilon: float = 0.25):
        self.k1, self.b, self.epsilon = k1, b, epsilon

    def fit(self, documents: list[str]) -> "SparseBM25":
        self.vectorizer = CountVectorizer(
            token_pattern=TOKEN_PATTERN, lowercase=False, dtype=np.float32,
        )
        counts = self.vectorizer.fit_transform(documents)
        self.doc_len = np.asarray(counts.sum(axis=1)).ravel().astype(np.float32)
        self.matrix = counts.tocsc()
        self.n_docs = counts.shape[0]
        del counts
        document_frequency = np.diff(self.matrix.indptr).astype(np.float64)
        idf = np.log((self.n_docs - document_frequency + 0.5) / (document_frequency + 0.5))
        idf[idf < 0] = self.epsilon * float(idf.mean())
        self.idf = idf.astype(np.float32)
        self.norm = self.k1 * (1 - self.b + self.b * self.doc_len / self.doc_len.mean())
        return self

    def top_k(self, query: str, allowed: np.ndarray, k: int = TOP_K) -> np.ndarray:
        scores = np.zeros(self.n_docs, dtype=np.float32)
        for token, query_tf in Counter(query.split()).items():
            feature = self.vectorizer.vocabulary_.get(token)
            if feature is None:
                continue
            start, stop = self.matrix.indptr[feature : feature + 2]
            rows, term_tf = self.matrix.indices[start:stop], self.matrix.data[start:stop]
            scores[rows] += query_tf * self.idf[feature] * term_tf * (self.k1 + 1) / (term_tf + self.norm[rows])
        return allowed[safe_top_k(scores[allowed], k)]


def rrf_fuse(left: np.ndarray, right: np.ndarray) -> np.ndarray:
    scores: dict[int, float] = {}
    for ranking in (left, right):
        for rank, item_idx in enumerate(ranking, start=1):
            scores[int(item_idx)] = scores.get(int(item_idx), 0.0) + 1.0 / (RRF_K + rank)
    return np.asarray(sorted(scores, key=lambda index: (-scores[index], index))[:TOP_K], dtype=np.int64)


lexical_started = time.perf_counter()
item_text_bm25 = compose_document_text(
    candidate_items, ("item_title_raw", "item_infm_params_text", "item_description_raw")
).map(stem_russian_text).tolist()
bm25 = SparseBM25().fit(item_text_bm25)
bm25_queries = [stem_russian_text(query) for query in benchmark_queries["search_query"]]
bm25_rankings = [
    bm25.top_k(query, allowed)
    for query, allowed in zip(bm25_queries, allowed_indices_by_query, strict=True)
]
del bm25, item_text_bm25
gc.collect()

# The retained M1 char source is control-normalized title text, with no
# stemming or stopword removal.
title_char_texts = candidate_items["item_title_raw"].map(normalize_russian_text).tolist()
char_vectorizer = TfidfVectorizer(
    analyzer="char_wb", lowercase=False, ngram_range=(3, 5), min_df=2,
    max_features=CHAR_MAX_FEATURES, sublinear_tf=True, dtype=np.float32,
)
char_matrix = char_vectorizer.fit_transform(title_char_texts)
char_query_matrix = char_vectorizer.transform(
    [normalize_russian_text(query) for query in benchmark_queries["search_query"]]
)
char_rankings: list[np.ndarray] = []
for start in range(0, char_query_matrix.shape[0], CHAR_BATCH_SIZE):
    stop = min(start + CHAR_BATCH_SIZE, char_query_matrix.shape[0])
    scores_batch = (char_query_matrix[start:stop] @ char_matrix.T).toarray()
    for scores, allowed in zip(scores_batch, allowed_indices_by_query[start:stop], strict=True):
        char_rankings.append(allowed[safe_top_k(scores[allowed], TOP_K)])
del char_matrix, char_query_matrix, char_vectorizer, title_char_texts
gc.collect()

m1_rankings = [
    item_ids[rrf_fuse(bm25_ranking, char_ranking)].astype(str).tolist()
    for bm25_ranking, char_ranking in zip(bm25_rankings, char_rankings, strict=True)
]
lexical_seconds = time.perf_counter() - lexical_started
assert len(m1_rankings) == len(benchmark_queries)
assert all(len(row) == len(set(row)) and len(row) <= TOP_K for row in m1_rankings)
print({"m1_queries": len(m1_rankings), "m1_seconds": round(lexical_seconds, 2)})


## 4. M2 nearest historical-query candidates

Historical interactions are limited to items in the benchmark corpus;
the nearest query is found with character TF-IDF and candidates are
filtered to the benchmark query category.


In [ ]:
def build_history_lookup(frame: pd.DataFrame) -> dict[str, list[str]]:
    # Support counts are logical query contexts, not duplicated click rows.
    unique = frame[["query_group", "query_text_norm", "item_id"]].drop_duplicates()
    counts = unique.groupby(["query_text_norm", "item_id"], sort=False).size().rename("support").reset_index()
    lookup: dict[str, list[str]] = {}
    for query_text, part in counts.groupby("query_text_norm", sort=False):
        part = part.sort_values(["support", "item_id"], ascending=[False, True])
        lookup[str(query_text)] = part["item_id"].astype(str).tolist()
    return lookup


def filter_history_by_category(candidates: list[str], category_key: str) -> list[str]:
    allowed = category_to_item_ids.get(str(category_key), item_id_set)
    return [item_id for item_id in candidates if item_id in allowed][:TOP_K]


history_started = time.perf_counter()
history_lookup = build_history_lookup(history_pairs)
history_texts = np.asarray(sorted(history_lookup), dtype=object)
assert len(history_texts) > 0
history_vectorizer = TfidfVectorizer(
    analyzer="char_wb", preprocessor=normalize_russian_text, lowercase=False,
    ngram_range=(3, 5), min_df=1, max_features=CHAR_MAX_FEATURES,
    sublinear_tf=True, dtype=np.float32,
)
history_matrix = history_vectorizer.fit_transform(history_texts)
benchmark_history_matrix = history_vectorizer.transform(benchmark_queries["query_text_norm"].tolist())
history_rankings: list[list[str]] = []
for start in range(0, benchmark_history_matrix.shape[0], CHAR_BATCH_SIZE):
    stop = min(start + CHAR_BATCH_SIZE, benchmark_history_matrix.shape[0])
    scores_batch = (benchmark_history_matrix[start:stop] @ history_matrix.T).toarray()
    for scores, category in zip(scores_batch, benchmark_queries["category_key"].iloc[start:stop], strict=True):
        best_index = int(np.argmax(scores))
        history_rankings.append(
            filter_history_by_category(history_lookup[str(history_texts[best_index])], str(category))
            if float(scores[best_index]) > 0 else []
        )
del history_matrix, benchmark_history_matrix, history_vectorizer
gc.collect()
history_seconds = time.perf_counter() - history_started

assert len(history_rankings) == len(benchmark_queries)
assert all(len(row) == len(set(row)) and len(row) <= TOP_K for row in history_rankings)
print({
    "m2_history_unique_queries": len(history_texts),
    "m2_history_coverage": round(float(np.mean([bool(row) for row in history_rankings])), 6),
    "m2_seconds": round(history_seconds, 2),
})


## 5. E5 direct dense candidates

With two visible GPUs, independent embedding batches are distributed
across both. The default model is `intfloat/multilingual-e5-large-instruct`.
To test a completed E08 model, attach its exported checkpoint as a
Kaggle Input and flip `USE_FINETUNED_E5` in the setup cell; no other
pipeline setting should change.


In [ ]:
def compose_e5_query_text(frame: pd.DataFrame) -> list[str]:
    query = frame["search_query"].fillna("").astype(str).str.strip()
    filters = frame["search_infm_params_text"].fillna("").astype(str).str.strip()
    raw_texts = [
        value if not params else f"{value}\nФильтры поиска: {params}"
        for value, params in zip(query, filters, strict=True)
    ]
    return [f"Instruct: {E5_QUERY_INSTRUCTION}\nQuery: {text}" for text in raw_texts]


def compose_e5_item_text(frame: pd.DataFrame) -> list[str]:
    title = frame["item_title_raw"].fillna("").astype(str).str.strip()
    params = frame["item_infm_params_text"].fillna("").astype(str).str.strip()
    description = (
        frame["item_description_raw"].fillna("").astype(str)
        .str.slice(stop=DESCRIPTION_CHAR_LIMIT).str.strip()
    )
    texts: list[str] = []
    for title_value, params_value, description_value in zip(title, params, description, strict=True):
        fields = [f"Название услуги: {title_value}"]
        if params_value:
            fields.append(f"Параметры: {params_value}")
        if description_value:
            fields.append(f"Описание: {description_value}")
        texts.append("\n".join(fields))
    return texts


def exact_category_dense_search(
    query_embeddings: np.ndarray, item_embeddings: np.ndarray,
) -> tuple[list[list[str]], int]:
    rankings: list[list[str]] = [[] for _ in range(len(query_embeddings))]
    positions_by_category: dict[str, list[int]] = {}
    for position, category in enumerate(benchmark_queries["search_category"].tolist()):
        positions_by_category.setdefault(str(category), []).append(position)
    fallback_queries = 0
    for category, positions in positions_by_category.items():
        allowed = category_to_indices.get(category)
        if allowed is None or len(allowed) == 0:
            allowed = all_item_indices
            fallback_queries += len(positions)
        k = min(TOP_K, len(allowed))
        for start in range(0, len(positions), DENSE_SCORE_BATCH_SIZE):
            batch_positions = positions[start : start + DENSE_SCORE_BATCH_SIZE]
            scores = query_embeddings[batch_positions] @ item_embeddings[allowed].T
            top_local = np.argpartition(scores, kth=scores.shape[1] - k, axis=1)[:, -k:]
            top_scores = np.take_along_axis(scores, top_local, axis=1)
            order = np.argsort(top_scores, axis=1)[:, ::-1]
            top_global = allowed[np.take_along_axis(top_local, order, axis=1)]
            for query_position, item_positions in zip(batch_positions, top_global, strict=True):
                rankings[query_position] = item_ids[item_positions].astype(str).tolist()
    return rankings, fallback_queries


def model_revision(model: object) -> str:
    candidates = [model]
    try:
        candidates.append(model._first_module())
    except (AttributeError, TypeError):
        pass
    for candidate in candidates:
        auto_model = getattr(candidate, "auto_model", None)
        config = getattr(auto_model, "config", None)
        revision = getattr(config, "_commit_hash", None)
        if revision:
            return str(revision)
    return "unavailable"


import torch
from sentence_transformers import SentenceTransformer

if REQUIRE_GPU and not torch.cuda.is_available():
    raise RuntimeError("Enable a Kaggle GPU accelerator before running M9.")
device = "cuda" if torch.cuda.is_available() else "cpu"
visible_devices = [f"cuda:{index}" for index in range(torch.cuda.device_count())] if device == "cuda" else []
encoding_devices = visible_devices if USE_ALL_VISIBLE_GPUS else visible_devices[:1]
multi_gpu_enabled = len(encoding_devices) > 1
if device == "cuda":
    assert encoding_devices, "PyTorch reported CUDA but no device is available."

dense_started = time.perf_counter()
e5_item_texts = compose_e5_item_text(candidate_items)
e5_query_texts = compose_e5_query_text(benchmark_queries)
model = SentenceTransformer(MODEL_SOURCE, device="cpu" if multi_gpu_enabled else device)
model.max_seq_length = 256
if device == "cuda" and not multi_gpu_enabled:
    model.half()
pool = None
item_embeddings = query_embeddings = None
try:
    resolved_model_revision = model_revision(model)
    if multi_gpu_enabled:
        # This is the same data-parallel encoder route that completed E05a.
        pool = model.start_multi_process_pool(target_devices=encoding_devices)
    encode_kwargs = {"batch_size": DENSE_BATCH_SIZE, "show_progress_bar": True,
                     "convert_to_numpy": True, "normalize_embeddings": True}
    if pool is not None:
        encode_kwargs.update({"pool": pool, "chunk_size": 1_000})
    item_started = time.perf_counter()
    item_embeddings = np.asarray(model.encode(e5_item_texts, **encode_kwargs), dtype=np.float32)
    dense_item_seconds = time.perf_counter() - item_started
    query_started = time.perf_counter()
    query_embeddings = np.asarray(model.encode(e5_query_texts, **encode_kwargs), dtype=np.float32)
    dense_query_seconds = time.perf_counter() - query_started
    dense_search_started = time.perf_counter()
    dense_rankings, dense_fallback_queries = exact_category_dense_search(query_embeddings, item_embeddings)
    dense_search_seconds = time.perf_counter() - dense_search_started
finally:
    if pool is not None:
        model.stop_multi_process_pool(pool)
    del model, item_embeddings, query_embeddings
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

dense_seconds = time.perf_counter() - dense_started
assert len(dense_rankings) == len(benchmark_queries)
assert all(len(row) == len(set(row)) and len(row) <= TOP_K for row in dense_rankings)
print({
    "model_source": MODEL_SOURCE,
    "model_revision": resolved_model_revision,
    "encoding_devices": encoding_devices or ["cpu"],
    "dense_item_seconds": round(dense_item_seconds, 2),
    "dense_query_seconds": round(dense_query_seconds, 2),
    "dense_search_seconds": round(dense_search_seconds, 2),
    "dense_total_seconds": round(dense_seconds, 2),
    "dense_category_fallback_queries": dense_fallback_queries,
})


## 6. Final fusion and `answer.csv`

Apply M6's frozen history-20 → E5-10 → M1-fill policy. The validation
below checks all query IDs, item IDs, duplicate rules, exact column
names and the 50-candidate budget before saving anything.


In [ ]:
def merge_candidates(history: list[str], dense: list[str], lexical: list[str]) -> list[str]:
    """Retained M6 policy: 20 history, then 10 E5, then M1 RRF to 50."""
    selected: list[str] = []
    for source, quota in ((history, HISTORY_QUOTA), (dense, DENSE_QUOTA), (lexical, FINAL_K)):
        for item_id in source[:quota]:
            if item_id not in selected:
                selected.append(item_id)
            if len(selected) == FINAL_K:
                return selected
    return selected[:FINAL_K]


final_started = time.perf_counter()
final_rankings = [
    merge_candidates(history, dense, lexical)
    for history, dense, lexical in zip(history_rankings, dense_rankings, m1_rankings, strict=True)
]

# Competition-format validation happens before persisting the submission.
item_id_pattern = re.compile(r"^[0-9a-f]{16}$")
assert len(final_rankings) == len(benchmark_queries)
assert all(0 < len(row) <= FINAL_K for row in final_rankings)
assert all(len(row) == len(set(row)) for row in final_rankings)
assert all(item_id in item_id_set for row in final_rankings for item_id in row)
assert all(item_id_pattern.fullmatch(item_id) for row in final_rankings for item_id in row)

answer = pd.DataFrame({
    "query_id": benchmark_queries["query_id"].astype(str),
    "answer": [" ".join(row) for row in final_rankings],
})
assert list(answer.columns) == ["query_id", "answer"]
assert answer["query_id"].is_unique and len(answer) == len(benchmark_queries)
assert all(len(value.split()) <= FINAL_K for value in answer["answer"])
ANSWER_PATH = OUTPUT_DIR / "answer.csv"
answer.to_csv(ANSWER_PATH, index=False, encoding="utf-8")
final_seconds = time.perf_counter() - final_started

print({
    "answer_path": str(ANSWER_PATH),
    "answer_rows": len(answer),
    "mean_candidates": round(float(np.mean([len(row) for row in final_rankings])), 3),
    "finalize_seconds": round(final_seconds, 2),
})
display(answer.head(3))


## 7. Kaggle Output, ZIP and ClearML artifacts

After this cell finishes, use **Save Version** and download
`answer.csv` or the ZIP from the Output pane. The archive stores audit
records but intentionally omits temporary full-corpus embeddings.


In [ ]:
def save_rankings_jsonl(path: Path, rankings: list[list[str]]) -> None:
    with path.open("w", encoding="utf-8") as handle:
        for query_id, candidates in zip(benchmark_queries["query_id"].astype(str), rankings, strict=True):
            handle.write(json.dumps({"query_id": query_id, "candidate_item_ids": candidates}, ensure_ascii=False) + "\n")


m1_path = OUTPUT_DIR / f"m1_rrf_top{TOP_K}.jsonl"
m2_path = OUTPUT_DIR / f"m2_nearest_history_top{TOP_K}.jsonl"
e5_path = OUTPUT_DIR / f"e5_direct_top{TOP_K}.jsonl"
final_path = OUTPUT_DIR / f"final_top{FINAL_K}.jsonl"
for path, rankings in ((m1_path, m1_rankings), (m2_path, history_rankings),
                       (e5_path, dense_rankings), (final_path, final_rankings)):
    save_rankings_jsonl(path, rankings)

timings = pd.DataFrame([
    {"stage": "load_data", "seconds": load_seconds},
    {"stage": "m1_lexical", "seconds": lexical_seconds},
    {"stage": "m2_history", "seconds": history_seconds},
    {"stage": "e5_item_encode", "seconds": dense_item_seconds},
    {"stage": "e5_query_encode", "seconds": dense_query_seconds},
    {"stage": "e5_category_search", "seconds": dense_search_seconds},
    {"stage": "e5_total", "seconds": dense_seconds},
    {"stage": "finalize", "seconds": final_seconds},
    {"stage": "notebook_total", "seconds": time.perf_counter() - NOTEBOOK_STARTED},
])
TIMINGS_PATH = OUTPUT_DIR / "timings.csv"
timings.to_csv(TIMINGS_PATH, index=False)

manifest = {
    "stage": "M9_final_submission",
    "pipeline": "M1 stemmed-BM25 + title-char-TFIDF RRF; M2 nearest history; E5 direct dense",
    "source_policy": {"history_quota": HISTORY_QUOTA, "dense_quota": DENSE_QUOTA, "final_k": FINAL_K},
    "model_source": MODEL_SOURCE,
    "model_revision": resolved_model_revision,
    "use_finetuned_e5": USE_FINETUNED_E5,
    "encoding_devices": encoding_devices or ["cpu"],
    "source_rows": {
        "benchmark_queries": int(len(benchmark_queries)),
        "candidate_items": int(len(candidate_items)),
        "train_positive_pairs": int(len(train_pairs)),
        "history_pairs_in_candidate_corpus": int(len(history_pairs)),
    },
    "category_fallback_queries": {
        "lexical_and_history": category_fallback_queries,
        "dense": int(dense_fallback_queries),
    },
    "submission_validation": {
        "answer_rows": int(len(answer)),
        "unique_query_ids": int(answer["query_id"].nunique()),
        "mean_candidates": float(np.mean([len(row) for row in final_rankings])),
        "all_item_ids_in_corpus": True,
        "no_duplicates_within_answer": True,
    },
    "source_files": {
        path.name: {"bytes": path.stat().st_size, "modified_ns": path.stat().st_mtime_ns}
        for path in (TRAIN_PATH, BENCHMARK_QUERIES_PATH, BENCHMARK_ITEMS_PATH)
    },
}
MANIFEST_PATH = OUTPUT_DIR / "m9_manifest.json"
MANIFEST_PATH.write_text(json.dumps(manifest, ensure_ascii=False, indent=2), encoding="utf-8")

for _, row in timings.iterrows():
    clearml_logger.report_scalar("M9 timing seconds", str(row["stage"]), float(row["seconds"]), 0)
clearml_logger.report_scalar("M9 mean candidates", "final", float(manifest["submission_validation"]["mean_candidates"]), 0)
clearml_logger.report_scalar("M9 answer rows", "final", float(len(answer)), 0)
clearml_logger.report_table("M9 timings", "seconds", 0, table_plot=timings)
clearml_task.set_parameter("results/answer_path", str(ANSWER_PATH))
clearml_task.set_parameter("results/answer_rows", int(len(answer)))
clearml_task.set_parameter("results/mean_candidates", float(manifest["submission_validation"]["mean_candidates"]))

for artifact_name, artifact_path in {
    "m9_answer_csv": ANSWER_PATH,
    "m9_manifest": MANIFEST_PATH,
    "m9_timings": TIMINGS_PATH,
    "m9_m1_top200": m1_path,
    "m9_m2_top200": m2_path,
    "m9_e5_top200": e5_path,
    "m9_final_top50": final_path,
}.items():
    clearml_task.upload_artifact(artifact_name, artifact_object=artifact_path)

ZIP_PATH = OUTPUT_DIR.with_suffix(".zip")
shutil.make_archive(str(ZIP_PATH.with_suffix("")), "zip", root_dir=str(OUTPUT_DIR.parent), base_dir=OUTPUT_DIR.name)
assert ZIP_PATH.exists()
clearml_task.upload_artifact("m9_submission_artifacts_zip", artifact_object=ZIP_PATH)
clearml_task.close()

from IPython.display import FileLink, display

print({"answer_csv": str(ANSWER_PATH), "zip_path": str(ZIP_PATH), "clearml_task_id": clearml_task.id})
display(FileLink(ANSWER_PATH))
display(FileLink(ZIP_PATH))


## Completion checklist

- Confirm the final cell printed the output path and a ClearML task ID.
- Create a Kaggle Version so `/kaggle/working` becomes downloadable.
- Submit only `answer.csv`, unchanged.
- If E08 wins proxy evaluation, change only the explicit checkpoint
  switch and run this same notebook again.
